Any arbitrary polygon (with any number of corners, including non-convex L-shapes or notched brackets) into Q4 quadrilateral elements—or a Quad-Dominant (Q4 + T3) hybrid mesh—using a standard Triangle-to-Quad (Catmull-Clark / Midpoint Subdivision or Triangle Pairing) algorithm.


Arbitrary Polygon Q4 & Hybrid (Q4 + T3) Meshing Works

• Method 1 — Triangle-to-Quad Merging (Quad-Dominant Q4 + T3): First generate an unstructured triangle (T3) mesh inside any polygon. Then greedily pair adjacent triangles that share an internal edge into well-shaped Q4 quadrilaterals, leaving any unpaired odd triangles as T3 elements.

• Method 2 — Midpoint Subdivision (Pure Q4 on Any Polygon): Start from any unstructured triangle mesh and split every triangle into 3 Q4 quadrilaterals by connecting its centroid to the midpoints of its 3 edges. Every resulting element is guaranteed to be a valid 4-node Q4 element, even on complex polygons!


In [2]:
import tkinter as tk
from tkinter import messagebox, ttk
import matplotlib.path as mpath
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.collections import PolyCollection
from matplotlib.figure import Figure
import numpy as np
from scipy.spatial import Delaunay


# =========================================================
# 1. BASE TRIANGULATION ENGINE (Works on ANY Polygon)
# =========================================================
def ensure_ccw(vertices):
    x, y = vertices[:, 0], vertices[:, 1]
    signed_area = 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)
    return vertices[::-1] if signed_area < 0 else vertices


def build_base_t3(raw_vertices, h_size, use_smoothing, iterations):
    """Generates a clean base T3 mesh inside any arbitrary polygon."""
    vertices = ensure_ccw(np.asarray(raw_vertices, dtype=float))

    # Seed outer boundary
    boundary_nodes = []
    num_edges = len(vertices)
    for i in range(num_edges):
        p_start = vertices[i]
        p_end = vertices[(i + 1) % num_edges]
        edge_len = np.linalg.norm(p_end - p_start)
        n_div = max(1, int(np.round(edge_len / h_size)))
        for k in range(n_div):
            t = k / n_div
            boundary_nodes.append((1.0 - t) * p_start + t * p_end)
    boundary_nodes = np.array(boundary_nodes)

    # Seed staggered interior nodes
    poly_path = mpath.Path(vertices)
    xmin, ymin = vertices.min(axis=0)
    xmax, ymax = vertices.max(axis=0)
    dy = h_size * np.sqrt(3) / 2.0

    interior_nodes = []
    row = 0
    y = ymin + dy
    while y < ymax:
        x_shift = (h_size / 2.0) if (row % 2 == 1) else 0.0
        x = xmin + (h_size / 2.0) + x_shift
        while x < xmax:
            pt = np.array([x, y])
            if poly_path.contains_point(pt):
                dist = np.min(np.linalg.norm(boundary_nodes - pt, axis=1))
                if dist > 0.55 * h_size:
                    interior_nodes.append(pt)
            x += h_size
        y += dy
        row += 1

    nodes = (
        np.vstack([boundary_nodes, np.array(interior_nodes)])
        if len(interior_nodes) > 0
        else boundary_nodes
    )

    # Delaunay + Clipping + CCW ordering
    tri = Delaunay(nodes)
    valid_tris = []
    for simplex in tri.simplices:
        pts = nodes[simplex]
        if poly_path.contains_point(pts.mean(axis=0)):
            area2 = (pts[1, 0] - pts[0, 0]) * (pts[2, 1] - pts[0, 1]) - (
                pts[1, 1] - pts[0, 1]
            ) * (pts[2, 0] - pts[0, 0])
            if area2 > 1e-12:
                valid_tris.append(simplex)
            elif area2 < -1e-12:
                valid_tris.append([simplex[0], simplex[2], simplex[1]])
    tris = np.array(valid_tris, dtype=int)

    # Smooth base T3 mesh if requested
    if use_smoothing and len(interior_nodes) > 0 and len(tris) > 0:
        nodes = apply_laplacian_smoothing(
            nodes, tris, set(range(len(boundary_nodes))), iterations
        )

    return nodes, tris, len(boundary_nodes)


def apply_laplacian_smoothing(nodes, elements, fixed_node_ids, iterations):
    """Smooths any mix of T3 and Q4 elements while keeping boundary nodes fixed."""
    smoothed = nodes.copy()
    neighbors = {i: set() for i in range(len(smoothed))}
    for el in elements:
        k = len(el)
        for i in range(k):
            neighbors[el[i]].update([el[(i - 1) % k], el[(i + 1) % k]])

    for _ in range(iterations):
        new_pos = smoothed.copy()
        for idx in range(len(smoothed)):
            if idx not in fixed_node_ids and neighbors[idx]:
                nbrs = list(neighbors[idx])
                new_pos[idx] = 0.5 * smoothed[idx] + 0.5 * np.mean(
                    smoothed[nbrs], axis=0
                )
        smoothed = new_pos
    return smoothed


# =========================================================
# 2. HYBRID QUAD-DOMINANT (Q4 + T3) MERGER
# =========================================================
def merge_t3_to_hybrid_q4_t3(nodes, tris):
    """Pairs adjacent triangles into Q4 elements; leaves unpaired ones as T3."""
    edge_to_tris = {}
    for t_idx, tri in enumerate(tris):
        for i in range(3):
            u, v = tri[i], tri[(i + 1) % 3]
            edge = tuple(sorted((u, v)))
            edge_to_tris.setdefault(edge, []).append(t_idx)

    # Score internal shared edges by how close the merged quad angles are to 90 deg
    candidate_pairs = []
    for edge, adj_tris in edge_to_tris.items():
        if len(adj_tris) == 2:
            t1, t2 = adj_tris
            quad = form_ccw_quad(nodes, tris[t1], tris[t2], edge)
            if quad is not None:
                quality = quad_angle_quality(nodes[quad])
                if quality > 0.35:  # Reject overly distorted or concave quads
                    candidate_pairs.append((quality, t1, t2, quad))

    # Greedily merge best-shaped pairs first
    candidate_pairs.sort(key=lambda x: x[0], reverse=True)
    used_tris = set()
    q4_elements = []

    for _, t1, t2, quad in candidate_pairs:
        if t1 not in used_tris and t2 not in used_tris:
            q4_elements.append(quad)
            used_tris.add(t1)
            used_tris.add(t2)

    # Remaining unpaired triangles stay as T3 elements
    t3_elements = [
        list(tris[i]) for i in range(len(tris)) if i not in used_tris
    ]
    return nodes, q4_elements, t3_elements


def form_ccw_quad(nodes, tri1, tri2, shared_edge):
    """Combines two triangles sharing `shared_edge` into a 4-node CCW quad."""
    u, v = shared_edge
    opp1 = [n for n in tri1 if n not in shared_edge][0]
    opp2 = [n for n in tri2 if n not in shared_edge][0]
    quad = [u, opp1, v, opp2]
    pts = nodes[quad]

    # Check convexity & CCW winding via cross products at all 4 corners
    crosses = []
    for i in range(4):
        v1 = pts[(i + 1) % 4] - pts[i]
        v2 = pts[(i + 2) % 4] - pts[(i + 1) % 4]
        crosses.append(v1[0] * v2[1] - v1[1] * v2[0])

    if all(c > 1e-12 for c in crosses):
        return quad
    if all(c < -1e-12 for c in crosses):
        return quad[::-1]
    return None  # Non-convex quad rejected


def quad_angle_quality(pts):
    """Returns 1.0 for a perfect 90° rectangle, decreasing toward 0 for skewed quads."""
    max_dev = 0.0
    for k in range(4):
        v1 = pts[(k - 1) % 4] - pts[k]
        v2 = pts[(k + 1) % 4] - pts[k]
        cos_t = np.clip(
            np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)),
            -1.0,
            1.0,
        )
        angle = np.degrees(np.arccos(cos_t))
        max_dev = max(max_dev, abs(90.0 - angle))
    return max(0.0, 1.0 - max_dev / 90.0)


# =========================================================
# 3. PURE ALL-QUAD (Q4) SUBDIVISION FOR ANY POLYGON
# =========================================================
def subdivide_t3_to_pure_q4(nodes, tris, num_boundary_nodes):
    """Splits every triangle into 3 pure Q4 elements via centroid-to-midpoint lines."""
    new_nodes = list(nodes)
    edge_midpoints = {}
    q4_elements = []
    boundary_midpoints = set()

    # Count edge occurrences to identify outer boundary edges
    edge_counts = {}
    for tri in tris:
        for i in range(3):
            edge = tuple(sorted((tri[i], tri[(i + 1) % 3])))
            edge_counts[edge] = edge_counts.get(edge, 0) + 1

    def get_midpoint(u, v):
        edge = tuple(sorted((u, v)))
        if edge not in edge_midpoints:
            mid_pt = 0.5 * (nodes[u] + nodes[v])
            mid_idx = len(new_nodes)
            new_nodes.append(mid_pt)
            edge_midpoints[edge] = mid_idx
            if edge_counts[edge] == 1:
                boundary_midpoints.add(mid_idx)
        return edge_midpoints[edge]

    for n0, n1, n2 in tris:
        m01 = get_midpoint(n0, n1)
        m12 = get_midpoint(n1, n2)
        m20 = get_midpoint(n2, n0)

        # Add centroid node
        c_idx = len(new_nodes)
        new_nodes.append((nodes[n0] + nodes[n1] + nodes[n2]) / 3.0)

        # Create 3 CCW Q4 elements per triangle
        q4_elements.append([n0, m01, c_idx, m20])
        q4_elements.append([n1, m12, c_idx, m01])
        q4_elements.append([n2, m20, c_idx, m12])

    fixed_ids = set(range(num_boundary_nodes)) | boundary_midpoints
    return np.array(new_nodes), q4_elements, fixed_ids


# =========================================================
# INTERACTIVE TKINTER GUI
# =========================================================
class PolygonQ4T3App:
    def __init__(self, root):
        self.root = root
        self.root.title('Any-Polygon Q4 & T3 Hybrid Mesher')
        self.root.geometry('1000x660')

        self.clicked_pts = []
        self.polygon_closed = False

        # --- Left Sidebar ---
        panel = ttk.Frame(root, padding=15)
        panel.pack(side=tk.LEFT, fill=tk.Y)

        ttk.Label(
            panel, text='1. Mesh Element Mode', font=('Arial', 12, 'bold')
        ).pack(anchor=tk.W, pady=(0, 5))

        self.mode_var = tk.StringVar(value='HYBRID')
        ttk.Radiobutton(
            panel,
            text='Hybrid Quad-Dominant (Q4 + T3)',
            variable=self.mode_var,
            value='HYBRID',
            command=self.on_param_change,
        ).pack(anchor=tk.W)
        ttk.Radiobutton(
            panel,
            text='Pure All-Quad (Q4 Only)',
            variable=self.mode_var,
            value='PURE_Q4',
            command=self.on_param_change,
        ).pack(anchor=tk.W)
        ttk.Radiobutton(
            panel,
            text='Unstructured Triangle (T3 Only)',
            variable=self.mode_var,
            value='T3',
            command=self.on_param_change,
        ).pack(anchor=tk.W, pady=(0, 10))

        ttk.Separator(panel, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=5)

        ttk.Label(
            panel, text='2. Draw Any Polygon', font=('Arial', 12, 'bold')
        ).pack(anchor=tk.W, pady=(8, 4))
        ttk.Label(
            panel,
            text='• Left-Click: Add corner (3+ pts)\n• Right-Click: Close & Mesh',
            justify=tk.LEFT,
        ).pack(anchor=tk.W, pady=(0, 8))

        btn_row = ttk.Frame(panel)
        btn_row.pack(fill=tk.X, pady=(0, 12))
        ttk.Button(
            btn_row, text='Close & Mesh', command=self.close_and_mesh
        ).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=(0, 4))
        ttk.Button(
            btn_row, text='Clear / New', command=self.clear_canvas
        ).pack(side=tk.RIGHT, expand=True, fill=tk.X, padx=(4, 0))

        ttk.Separator(panel, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=5)

        ttk.Label(
            panel, text='3. Mesh Settings', font=('Arial', 12, 'bold')
        ).pack(anchor=tk.W, pady=(8, 4))

        ttk.Label(panel, text='Target Element Size (h):').pack(anchor=tk.W)
        self.h_var = tk.DoubleVar(value=0.60)
        ttk.Scale(
            panel,
            from_=0.25,
            to=1.20,
            variable=self.h_var,
            command=self.on_param_change,
        ).pack(fill=tk.X, pady=(2, 8))

        self.smooth_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(
            panel,
            text='Enable Laplacian Smoothing',
            variable=self.smooth_var,
            command=self.on_param_change,
        ).pack(anchor=tk.W, pady=(2, 6))

        ttk.Label(panel, text='Smoothing Iterations:').pack(anchor=tk.W)
        self.iter_var = tk.IntVar(value=10)
        ttk.Scale(
            panel,
            from_=1,
            to=30,
            variable=self.iter_var,
            command=self.on_param_change,
        ).pack(fill=tk.X, pady=(2, 12))

        ttk.Separator(panel, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=5)

        ttk.Label(
            panel, text='4. Mesh Summary', font=('Arial', 12, 'bold')
        ).pack(anchor=tk.W, pady=(8, 4))
        self.stats_label = ttk.Label(
            panel,
            text='Click any polygon on grid\nto generate Q4 / T3 mesh.',
            font=('Consolas', 10),
            justify=tk.LEFT,
        )
        self.stats_label.pack(anchor=tk.W)

        # --- Right Canvas ---
        plot_frame = ttk.Frame(root, padding=10)
        plot_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        self.fig = Figure(figsize=(6, 6), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        self.canvas.mpl_connect('button_press_event', self.on_canvas_click)

        self.redraw_drawing_state()

    def setup_axes(self, title):
        self.ax.clear()
        self.ax.set_xlim(0.0, 6.0)
        self.ax.set_ylim(0.0, 6.0)
        self.ax.set_aspect('equal', adjustable='box')
        self.ax.grid(True, linestyle='--', alpha=0.5)
        self.ax.set_xlabel('X')
        self.ax.set_ylabel('Y')
        self.ax.set_title(title)

    def on_canvas_click(self, event):
        if event.inaxes != self.ax or self.polygon_closed:
            return
        if event.button == 1:
            x_snap = round(event.xdata * 20) / 20.0
            y_snap = round(event.ydata * 20) / 20.0
            self.clicked_pts.append([x_snap, y_snap])
            self.redraw_drawing_state()
        elif event.button == 3:
            self.close_and_mesh()

    def redraw_drawing_state(self):
        self.setup_axes(
            f'Left-Click Polygon Corners ({len(self.clicked_pts)} placed) | Right-Click to Mesh'
        )
        if len(self.clicked_pts) > 0:
            pts = np.array(self.clicked_pts)
            self.ax.plot(pts[:, 0], pts[:, 1], 'k-o', lw=1.5, markersize=5)
            self.ax.plot(pts[0, 0], pts[0, 1], 'go', markersize=8)
        self.canvas.draw()

    def close_and_mesh(self):
        if len(self.clicked_pts) < 3:
            messagebox.showwarning(
                'Not Enough Points', 'Click at least 3 corners first.'
            )
            return
        self.polygon_closed = True
        self.update_mesh()

    def clear_canvas(self):
        self.clicked_pts = []
        self.polygon_closed = False
        self.stats_label.config(
            text='Click any polygon on grid\nto generate Q4 / T3 mesh.'
        )
        self.redraw_drawing_state()

    def on_param_change(self, _event=None):
        if self.polygon_closed:
            self.update_mesh()

    def update_mesh(self):
        h = round(self.h_var.get(), 2)
        use_smooth = self.smooth_var.get()
        iters = int(self.iter_var.get())
        mode = self.mode_var.get()

        nodes, tris, num_b = build_base_t3(
            self.clicked_pts, h, use_smooth, iters
        )

        if mode == 'T3':
            q4_els, t3_els = [], [list(t) for t in tris]
        elif mode == 'HYBRID':
            nodes, q4_els, t3_els = merge_t3_to_hybrid_q4_t3(nodes, tris)
            if use_smooth:
                nodes = apply_laplacian_smoothing(
                    nodes, q4_els + t3_els, set(range(num_b)), iters
                )
        else:  # PURE_Q4
            nodes, q4_els, fixed_ids = subdivide_t3_to_pure_q4(
                nodes, tris, num_b
            )
            t3_els = []
            if use_smooth:
                nodes = apply_laplacian_smoothing(
                    nodes, q4_els, fixed_ids, iters
                )

        total_els = len(q4_els) + len(t3_els)
        q4_pct = (100.0 * len(q4_els) / total_els) if total_els > 0 else 0.0

        report = (
            f'Polygon Corners:  {len(self.clicked_pts)}\n'
            f'Element Size (h): {h:.2f}\n'
            f'-----------------------------\n'
            f'Total Nodes:      {len(nodes)}\n'
            f'Q4 Quads (Blue):  {len(q4_els)} ({q4_pct:.0f}%)\n'
            f'T3 Tris (Amber):  {len(t3_els)}\n'
            f'Total Elements:   {total_els}\n'
            f'Reissner-Mindlin: {len(nodes) * 3} DOFs'
        )
        self.stats_label.config(text=report)

        self.setup_axes(f'Polygon Mesh: {len(q4_els)} Q4 + {len(t3_els)} T3')
        if q4_els:
            q4_polys = [nodes[el] for el in q4_els]
            self.ax.add_collection(
                PolyCollection(
                    q4_polys,
                    facecolors='#d6eaf8',
                    edgecolors='#1b4f72',
                    linewidths=0.9,
                )
            )
        if t3_els:
            t3_polys = [nodes[el] for el in t3_els]
            self.ax.add_collection(
                PolyCollection(
                    t3_polys,
                    facecolors='#fcf3cf',
                    edgecolors='#b7950b',
                    linewidths=0.9,
                )
            )
        self.ax.plot(nodes[:, 0], nodes[:, 1], 'ro', markersize=2.2)
        self.canvas.draw()


if __name__ == '__main__':
    root = tk.Tk()
    app = PolygonQ4T3App(root)
    root.mainloop()



 Changed in This Upgrade  local  foce  mesh  

• Global Node Density Controls: Added both a slider and a direct numerical spinbox (h) + "Remesh Now" button so you can dial in exact node spacing (0.15 to 1.20) and immediately regenerate the mesh without redrawing your polygon.

• Local Refinement Mode (Attractor Point): Added a "Click Refinement Focus" toggle. When enabled, clicking anywhere inside your meshed polygon places a star marker (★) that locally shrinks h near that point (controlled by a Refinement Factor slider) and automatically remeshes.

• Adaptive Interior & Boundary Seeding: Updated build_base_t3 so both boundary edge divisions and interior node spacing follow the spatially varying target size h(x, y).


In [5]:
import tkinter as tk
from tkinter import messagebox, ttk
import matplotlib.path as mpath
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.collections import PolyCollection
from matplotlib.figure import Figure
import numpy as np
from scipy.spatial import Delaunay


# =========================================================
# 1. ADAPTIVE NODE DENSITY & BASE T3 MESHER
# =========================================================
def ensure_ccw(vertices):
    x, y = vertices[:, 0], vertices[:, 1]
    signed_area = 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)
    return vertices[::-1] if signed_area < 0 else vertices


def local_target_size(pt, h_base, focus_pt, refine_strength):
    """Computes target element size h(x, y) at a given point."""
    if focus_pt is None or refine_strength <= 1.0:
        return h_base
    dist = np.linalg.norm(pt - focus_pt)
    # Smooth Gaussian-like transition from dense at focus_pt to h_base far away
    radius = 1.5
    factor = 1.0 + (refine_strength - 1.0) * np.exp(-((dist / radius) ** 2))
    return h_base / factor


def build_base_t3(
    raw_vertices,
    h_base,
    use_smoothing,
    iterations,
    focus_pt=None,
    refine_strength=1.0,
):
    """Generates a T3 mesh with global + optional local node density."""
    vertices = ensure_ccw(np.asarray(raw_vertices, dtype=float))

    # Step 2: Adaptive Boundary Seeding (marching along each edge using local h)
    boundary_nodes = []
    num_edges = len(vertices)
    for i in range(num_edges):
        p_start = vertices[i]
        p_end = vertices[(i + 1) % num_edges]
        edge_vec = p_end - p_start
        edge_len = np.linalg.norm(edge_vec)
        if edge_len < 1e-9:
            continue
        direction = edge_vec / edge_len

        # Walk from p_start toward p_end adapting step size to local_target_size
        pts_on_edge = [p_start]
        curr_dist = 0.0
        while True:
            curr_pt = p_start + curr_dist * direction
            step = local_target_size(curr_pt, h_base, focus_pt, refine_strength)
            if curr_dist + step >= edge_len - 0.35 * step:
                break
            curr_dist += step
            pts_on_edge.append(p_start + curr_dist * direction)

        # Rescale marching steps slightly so nodes hit p_end cleanly
        if len(pts_on_edge) > 1:
            t_vals = np.linspace(0.0, 1.0, len(pts_on_edge) + 1)[:-1]
            # Preserve relative spacing density along the edge
            raw_dists = [np.linalg.norm(p - p_start) for p in pts_on_edge]
            scaled_dists = np.array(raw_dists) * (
                edge_len / (raw_dists[-1] + step)
            )
            for d in scaled_dists:
                boundary_nodes.append(p_start + d * direction)
        else:
            boundary_nodes.append(p_start)

    boundary_nodes = np.array(boundary_nodes)

    # Step 3: Adaptive Interior Node Generation
    poly_path = mpath.Path(vertices)
    xmin, ymin = vertices.min(axis=0)
    xmax, ymax = vertices.max(axis=0)

    # Fine candidate grid filtered by local target spacing h(x, y)
    h_min = h_base / max(1.0, refine_strength)
    dy = h_min * np.sqrt(3) / 2.0

    accepted_nodes = list(boundary_nodes)
    row = 0
    y = ymin + dy
    while y < ymax:
        x_shift = (h_min / 2.0) if (row % 2 == 1) else 0.0
        x = xmin + (h_min / 2.0) + x_shift
        while x < xmax:
            pt = np.array([x, y])
            if poly_path.contains_point(pt):
                h_local = local_target_size(
                    pt, h_base, focus_pt, refine_strength
                )
                existing = np.array(accepted_nodes)
                min_dist = np.min(np.linalg.norm(existing - pt, axis=1))
                # Only keep point if it respects the local spacing h(x, y)
                if min_dist >= 0.78 * h_local:
                    accepted_nodes.append(pt)
            x += h_min
        y += dy
        row += 1

    nodes = np.array(accepted_nodes)
    num_b = len(boundary_nodes)

    # Step 4: Delaunay Triangulation + Polygon Clipping + CCW Ordering
    tri = Delaunay(nodes)
    valid_tris = []
    for simplex in tri.simplices:
        pts = nodes[simplex]
        if poly_path.contains_point(pts.mean(axis=0)):
            area2 = (pts[1, 0] - pts[0, 0]) * (pts[2, 1] - pts[0, 1]) - (
                pts[1, 1] - pts[0, 1]
            ) * (pts[2, 0] - pts[0, 0])
            if area2 > 1e-12:
                valid_tris.append(simplex)
            elif area2 < -1e-12:
                valid_tris.append([simplex[0], simplex[2], simplex[1]])
    tris = np.array(valid_tris, dtype=int)

    # Step 4.5: Optional Laplacian Smoothing
    if use_smoothing and len(nodes) > num_b and len(tris) > 0:
        nodes = apply_laplacian_smoothing(
            nodes, tris, set(range(num_b)), iterations
        )

    return nodes, tris, num_b


def apply_laplacian_smoothing(nodes, elements, fixed_node_ids, iterations):
    smoothed = nodes.copy()
    neighbors = {i: set() for i in range(len(smoothed))}
    for el in elements:
        k = len(el)
        for i in range(k):
            neighbors[el[i]].update([el[(i - 1) % k], el[(i + 1) % k]])

    for _ in range(iterations):
        new_pos = smoothed.copy()
        for idx in range(len(smoothed)):
            if idx not in fixed_node_ids and neighbors[idx]:
                nbrs = list(neighbors[idx])
                new_pos[idx] = 0.5 * smoothed[idx] + 0.5 * np.mean(
                    smoothed[nbrs], axis=0
                )
        smoothed = new_pos
    return smoothed


# =========================================================
# 2. HYBRID (Q4 + T3) & PURE Q4 CONVERTERS
# =========================================================
def merge_t3_to_hybrid_q4_t3(nodes, tris):
    edge_to_tris = {}
    for t_idx, tri in enumerate(tris):
        for i in range(3):
            edge = tuple(sorted((tri[i], tri[(i + 1) % 3])))
            edge_to_tris.setdefault(edge, []).append(t_idx)

    candidate_pairs = []
    for edge, adj_tris in edge_to_tris.items():
        if len(adj_tris) == 2:
            t1, t2 = adj_tris
            quad = form_ccw_quad(nodes, tris[t1], tris[t2], edge)
            if quad is not None:
                quality = quad_angle_quality(nodes[quad])
                if quality > 0.35:
                    candidate_pairs.append((quality, t1, t2, quad))

    candidate_pairs.sort(key=lambda x: x[0], reverse=True)
    used_tris = set()
    q4_elements = []
    for _, t1, t2, quad in candidate_pairs:
        if t1 not in used_tris and t2 not in used_tris:
            q4_elements.append(quad)
            used_tris.add(t1)
            used_tris.add(t2)

    t3_elements = [
        list(tris[i]) for i in range(len(tris)) if i not in used_tris
    ]
    return nodes, q4_elements, t3_elements


def form_ccw_quad(nodes, tri1, tri2, shared_edge):
    u, v = shared_edge
    opp1 = [n for n in tri1 if n not in shared_edge][0]
    opp2 = [n for n in tri2 if n not in shared_edge][0]
    quad = [u, opp1, v, opp2]
    pts = nodes[quad]

    crosses = []
    for i in range(4):
        v1 = pts[(i + 1) % 4] - pts[i]
        v2 = pts[(i + 2) % 4] - pts[(i + 1) % 4]
        crosses.append(v1[0] * v2[1] - v1[1] * v2[0])

    if all(c > 1e-12 for c in crosses):
        return quad
    if all(c < -1e-12 for c in crosses):
        return quad[::-1]
    return None


def quad_angle_quality(pts):
    max_dev = 0.0
    for k in range(4):
        v1 = pts[(k - 1) % 4] - pts[k]
        v2 = pts[(k + 1) % 4] - pts[k]
        cos_t = np.clip(
            np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)),
            -1.0,
            1.0,
        )
        max_dev = max(max_dev, abs(90.0 - np.degrees(np.arccos(cos_t))))
    return max(0.0, 1.0 - max_dev / 90.0)


def subdivide_t3_to_pure_q4(nodes, tris, num_boundary_nodes):
    new_nodes = list(nodes)
    edge_midpoints = {}
    q4_elements = []
    boundary_midpoints = set()

    edge_counts = {}
    for tri in tris:
        for i in range(3):
            edge = tuple(sorted((tri[i], tri[(i + 1) % 3])))
            edge_counts[edge] = edge_counts.get(edge, 0) + 1

    def get_midpoint(u, v):
        edge = tuple(sorted((u, v)))
        if edge not in edge_midpoints:
            mid_pt = 0.5 * (nodes[u] + nodes[v])
            mid_idx = len(new_nodes)
            new_nodes.append(mid_pt)
            edge_midpoints[edge] = mid_idx
            if edge_counts[edge] == 1:
                boundary_midpoints.add(mid_idx)
        return edge_midpoints[edge]

    for n0, n1, n2 in tris:
        m01 = get_midpoint(n0, n1)
        m12 = get_midpoint(n1, n2)
        m20 = get_midpoint(n2, n0)
        c_idx = len(new_nodes)
        new_nodes.append((nodes[n0] + nodes[n1] + nodes[n2]) / 3.0)
        q4_elements.append([n0, m01, c_idx, m20])
        q4_elements.append([n1, m12, c_idx, m01])
        q4_elements.append([n2, m20, c_idx, m12])

    fixed_ids = set(range(num_boundary_nodes)) | boundary_midpoints
    return np.array(new_nodes), q4_elements, fixed_ids


# =========================================================
# 3. INTERACTIVE TKINTER REMESHING APPLICATION
# =========================================================
class RemeshDensityApp:
    def __init__(self, root):
        self.root = root
        self.root.title('Interactive Remesher: Global & Local Node Density')
        self.root.geometry('1020x690')

        self.clicked_pts = []
        self.polygon_closed = False
        self.focus_pt = None  # Local refinement attractor point [x, y]

        # --- Left Sidebar ---
        panel = ttk.Frame(root, padding=14)
        panel.pack(side=tk.LEFT, fill=tk.Y)

        # 1. Element Mode
        ttk.Label(
            panel, text='1. Element Type', font=('Arial', 11, 'bold')
        ).pack(anchor=tk.W, pady=(0, 4))

        self.mode_var = tk.StringVar(value='HYBRID')
        for label, val in [
            ('Hybrid Quad-Dominant (Q4 + T3)', 'HYBRID'),
            ('Pure All-Quad (Q4 Only)', 'PURE_Q4'),
            ('Triangle (T3 Only)', 'T3'),
        ]:
            ttk.Radiobutton(
                panel,
                text=label,
                variable=self.mode_var,
                value=val,
                command=self.remesh,
            ).pack(anchor=tk.W)

        ttk.Separator(panel, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=6)

        # 2. Geometry Drawing
        ttk.Label(
            panel, text='2. Polygon Geometry', font=('Arial', 11, 'bold')
        ).pack(anchor=tk.W, pady=(2, 4))
        ttk.Label(
            panel,
            text='• Left-Click: Add corners\n• Right-Click: Close & Mesh',
            justify=tk.LEFT,
        ).pack(anchor=tk.W, pady=(0, 6))

        btn_row = ttk.Frame(panel)
        btn_row.pack(fill=tk.X, pady=(0, 8))
        ttk.Button(
            btn_row, text='Close & Mesh', command=self.close_and_mesh
        ).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=(0, 3))
        ttk.Button(
            btn_row, text='Clear All', command=self.clear_canvas
        ).pack(side=tk.RIGHT, expand=True, fill=tk.X, padx=(3, 0))

        ttk.Separator(panel, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=6)

        # 3. Global Node Density & Remesh
        ttk.Label(
            panel, text='3. Global Node Density', font=('Arial', 11, 'bold')
        ).pack(anchor=tk.W, pady=(2, 4))

        h_row = ttk.Frame(panel)
        h_row.pack(fill=tk.X, pady=(0, 4))
        ttk.Label(h_row, text='Base Spacing (h):').pack(side=tk.LEFT)
        self.h_var = tk.DoubleVar(value=0.55)
        self.h_spin = ttk.Spinbox(
            h_row,
            from_=0.15,
            to=1.50,
            increment=0.05,
            textvariable=self.h_var,
            width=6,
            command=self.remesh,
        )
        self.h_spin.pack(side=tk.RIGHT)

        ttk.Scale(
            panel,
            from_=0.15,
            to=1.20,
            variable=self.h_var,
            command=self.on_slider_drag,
        ).pack(fill=tk.X, pady=(0, 6))

        # Preset Density Quick Buttons
        preset_row = ttk.Frame(panel)
        preset_row.pack(fill=tk.X, pady=(0, 8))
        ttk.Button(
            preset_row,
            text='Coarse (0.8)',
            command=lambda: self.set_density(0.80),
        ).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=1)
        ttk.Button(
            preset_row,
            text='Medium (0.5)',
            command=lambda: self.set_density(0.50),
        ).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=1)
        ttk.Button(
            preset_row,
            text='Fine (0.28)',
            command=lambda: self.set_density(0.28),
        ).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=1)

        # 4. Local Density Refinement (Attractor Point)
        ttk.Label(
            panel,
            text='4. Local Density Refinement',
            font=('Arial', 11, 'bold'),
        ).pack(anchor=tk.W, pady=(4, 4))
        self.pick_focus_var = tk.BooleanVar(value=False)
        ttk.Checkbutton(
            panel,
            text='Click Mesh to Set Focus Point (★)',
            variable=self.pick_focus_var,
        ).pack(anchor=tk.W)

        ttk.Label(panel, text='Local Refinement Multiplier:').pack(
            anchor=tk.W, pady=(4, 0)
        )
        self.refine_var = tk.DoubleVar(value=2.2)
        ttk.Scale(
            panel,
            from_=1.0,
            to=4.0,
            variable=self.refine_var,
            command=self.on_slider_drag,
        ).pack(fill=tk.X, pady=(0, 6))

        ref_btn_row = ttk.Frame(panel)
        ref_btn_row.pack(fill=tk.X, pady=(0, 8))
        ttk.Button(
            ref_btn_row, text='Clear Focus Point', command=self.clear_focus
        ).pack(side=tk.LEFT, expand=True, fill=tk.X, padx=(0, 3))
        ttk.Button(
            ref_btn_row, text='Remesh Now', command=self.remesh
        ).pack(side=tk.RIGHT, expand=True, fill=tk.X, padx=(3, 0))

        # 5. Smoothing
        self.smooth_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(
            panel,
            text='Laplacian Smoothing (10 passes)',
            variable=self.smooth_var,
            command=self.remesh,
        ).pack(anchor=tk.W, pady=(2, 6))

        ttk.Separator(panel, orient=tk.HORIZONTAL).pack(fill=tk.X, pady=6)

        # 6. Summary Report
        self.stats_label = ttk.Label(
            panel,
            text='Draw a closed polygon\nto start remeshing.',
            font=('Consolas', 10),
            justify=tk.LEFT,
        )
        self.stats_label.pack(anchor=tk.W)

        # --- Right Canvas ---
        plot_frame = ttk.Frame(root, padding=10)
        plot_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        self.fig = Figure(figsize=(6, 6), dpi=100)
        self.ax = self.fig.add_subplot(111)
        self.canvas = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        self.canvas.mpl_connect('button_press_event', self.on_canvas_click)

        self.redraw_drawing_state()

    def setup_axes(self, title):
        self.ax.clear()
        self.ax.set_xlim(0.0, 6.0)
        self.ax.set_ylim(0.0, 6.0)
        self.ax.set_aspect('equal', adjustable='box')
        self.ax.grid(True, linestyle='--', alpha=0.5)
        self.ax.set_xlabel('X')
        self.ax.set_ylabel('Y')
        self.ax.set_title(title)

    def on_canvas_click(self, event):
        if event.inaxes != self.ax:
            return

        # Mode A: Drawing polygon corners
        if not self.polygon_closed:
            if event.button == 1:
                x_snap = round(event.xdata * 20) / 20.0
                y_snap = round(event.ydata * 20) / 20.0
                self.clicked_pts.append([x_snap, y_snap])
                self.redraw_drawing_state()
            elif event.button == 3:
                self.close_and_mesh()
            return

        # Mode B: Setting local refinement attractor point on an existing mesh
        if self.polygon_closed and self.pick_focus_var.get() and event.button == 1:
            self.focus_pt = np.array([event.xdata, event.ydata])
            self.remesh()

    def redraw_drawing_state(self):
        self.setup_axes(
            f'Left-Click Corners ({len(self.clicked_pts)} placed) | Right-Click to Mesh'
        )
        if len(self.clicked_pts) > 0:
            pts = np.array(self.clicked_pts)
            self.ax.plot(pts[:, 0], pts[:, 1], 'k-o', lw=1.5, markersize=5)
            self.ax.plot(pts[0, 0], pts[0, 1], 'go', markersize=8)
        self.canvas.draw()

    def close_and_mesh(self):
        if len(self.clicked_pts) < 3:
            messagebox.showwarning(
                'Not Enough Points', 'Click at least 3 corners first.'
            )
            return
        self.polygon_closed = True
        self.remesh()

    def clear_canvas(self):
        self.clicked_pts = []
        self.polygon_closed = False
        self.focus_pt = None
        self.stats_label.config(
            text='Draw a closed polygon\nto start remeshing.'
        )
        self.redraw_drawing_state()

    def clear_focus(self):
        self.focus_pt = None
        self.remesh()

    def set_density(self, h_val):
        self.h_var.set(h_val)
        self.remesh()

    def on_slider_drag(self, _event=None):
        self.h_var.set(round(self.h_var.get(), 2))
        self.remesh()

    def remesh(self):
        if not self.polygon_closed:
            return

        h_base = max(0.12, float(self.h_var.get()))
        use_smooth = self.smooth_var.get()
        mode = self.mode_var.get()
        refine_mult = float(self.refine_var.get())

        nodes, tris, num_b = build_base_t3(
            self.clicked_pts,
            h_base,
            use_smooth,
            iterations=10,
            focus_pt=self.focus_pt,
            refine_strength=refine_mult,
        )

        if mode == 'T3':
            q4_els, t3_els = [], [list(t) for t in tris]
        elif mode == 'HYBRID':
            nodes, q4_els, t3_els = merge_t3_to_hybrid_q4_t3(nodes, tris)
            if use_smooth:
                nodes = apply_laplacian_smoothing(
                    nodes, q4_els + t3_els, set(range(num_b)), 10
                )
        else:  # PURE_Q4
            nodes, q4_els, fixed_ids = subdivide_t3_to_pure_q4(
                nodes, tris, num_b
            )
            t3_els = []
            if use_smooth:
                nodes = apply_laplacian_smoothing(nodes, q4_els, fixed_ids, 10)

        total_els = len(q4_els) + len(t3_els)
        focus_str = (
            f'({self.focus_pt[0]:.2f}, {self.focus_pt[1]:.2f})'
            if self.focus_pt is not None
            else 'None'
        )

        report = (
            f'Base Spacing (h): {h_base:.2f}\n'
            f'Local Focus:      {focus_str}\n'
            f'-----------------------------\n'
            f'Total Nodes:      {len(nodes)}\n'
            f'Q4 Quads (Blue):  {len(q4_els)}\n'
            f'T3 Tris (Amber):  {len(t3_els)}\n'
            f'Total Elements:   {total_els}\n'
            f'Reissner-Mindlin: {len(nodes) * 3} DOFs'
        )
        self.stats_label.config(text=report)

        self.setup_axes(
            f'Remeshed: {len(nodes)} Nodes ({len(q4_els)} Q4 + {len(t3_els)} T3)'
        )
        if q4_els:
            self.ax.add_collection(
                PolyCollection(
                    [nodes[el] for el in q4_els],
                    facecolors='#d6eaf8',
                    edgecolors='#1b4f72',
                    linewidths=0.85,
                )
            )
        if t3_els:
            self.ax.add_collection(
                PolyCollection(
                    [nodes[el] for el in t3_els],
                    facecolors='#fcf3cf',
                    edgecolors='#b7950b',
                    linewidths=0.85,
                )
            )
        self.ax.plot(nodes[:, 0], nodes[:, 1], 'ro', markersize=2.0)

        # Draw star marker if a local density focus point is active
        if self.focus_pt is not None:
            self.ax.plot(
                self.focus_pt[0],
                self.focus_pt[1],
                marker='*',
                color='magenta',
                markersize=14,
                markeredgecolor='black',
                label='Refinement Focus',
            )
            self.ax.legend(loc='upper right')

        self.canvas.draw()


if __name__ == '__main__':
    root = tk.Tk()
    app = RemeshDensityApp(root)
    root.mainloop()
